# Order-parameter profiles of the reference systems (Fig. 6 a, b)

Pure DOPC and DOPC/CHOL 60/40, all chains (no classification), both leaflets and all frames. The order parameters are read from the raw feature tables written by `training_df_generation` (`dataframes/sym/{system}_{chain}_{leaflet}_op_ft_raw.csv`, columns C1–C16 = carbons C2–C17).

In [ ]:
import sys
sys.path.insert(0, "..")          # code_libraries is in the repository root; data in ../heavy_data_NO_GIT

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
from matplotlib.container import ErrorbarContainer

dataframe_dir = "../heavy_data_NO_GIT/dataframes/sym"
out_dir = "../heavy_data_NO_GIT/results/op_profiles"
os.makedirs(out_dir, exist_ok=True)
CHAINS = ["sn1", "sn2"]
carbon_columns = [f"C{i}" for i in range(1, 17)]          # C1..C16 in the tables = carbons C2..C17

In [ ]:
# mean S_CH per carbon over all chains, frames and both leaflets
REF_OP = {}
for sys_name, prefix in (("dopc_pure", "pure_dopc"), ("dopc_60", "dopc_60")):
    REF_OP[sys_name] = {}
    for chain in CHAINS:
        ops = pd.concat([pd.read_csv(os.path.join(dataframe_dir, f"{prefix}_{chain}_{leaflet}_op_ft_raw.csv"),
                                     usecols=carbon_columns) for leaflet in ("lower", "upper")])
        REF_OP[sys_name][chain] = ops.mean().to_numpy()
        print(sys_name, chain, np.round(REF_OP[sys_name][chain], 4))

In [ ]:
# paper figure style: bold text, ticks on all four sides pointing in, minor ticks
PAPER_STYLE = {"font.weight": "bold", "axes.labelweight": "bold", "axes.titleweight": "bold", "axes.linewidth": 2.5,
               "mathtext.default": "bf", "xtick.top": True, "ytick.right": True, "xtick.direction": "in",
               "ytick.direction": "in", "xtick.major.size": 10, "ytick.major.size": 10, "xtick.major.width": 2.5,
               "ytick.major.width": 2.5, "xtick.minor.size": 5, "ytick.minor.size": 5, "xtick.minor.width": 1.8,
               "ytick.minor.width": 1.8, "xtick.major.pad": 8, "ytick.major.pad": 8}
SN = {"sn1": r"$\mathbfit{sn}$-1", "sn2": r"$\mathbfit{sn}$-2"}
LO, LD = r"$\mathbf{L_o}$", r"$\mathbf{L_d}$"


def style_axes(ax, size=20):
    ax.xaxis.set_minor_locator(AutoMinorLocator(2))
    ax.yaxis.set_minor_locator(AutoMinorLocator(2))
    ax.tick_params(axis="both", which="both", labelsize=size, top=True, right=True, direction="in")
    for lab in ax.get_xticklabels() + ax.get_yticklabels():
        lab.set_fontweight("bold")


def legend_without_error_bars(ax, **kwargs):
    """Legend that shows only the line and marker of each curve, not its error bar."""
    handles, labels = ax.get_legend_handles_labels()
    handles = [h[0] if isinstance(h, ErrorbarContainer) else h for h in handles]
    return ax.legend(handles, labels, **kwargs)


ERR = dict(capsize=8, elinewidth=2.5, capthick=2.5, markersize=10)
# dopc_60 = Lo reference (red), dopc_pure = Ld reference (green)
REF_STYLE = {"dopc_60": (r"DOPC/CHOL=60/40", "red"), "dopc_pure": (r"PureDOPC", "green")}
carbons = np.arange(2, 18)
tick_carbons = carbons[carbons % 2 == 0]

with plt.rc_context(PAPER_STYLE):
    fig, axes = plt.subplots(1, 2, figsize=(30, 9), sharey=True, squeeze=False)
    for col, chain in enumerate(CHAINS):
        ax = axes[0, col]
        for sys_name, (label, color) in REF_STYLE.items():
            ax.plot(carbons, REF_OP[sys_name][chain], "o-", color=color, linewidth=4, markersize=ERR["markersize"], label=label)
        ax.set_title(f"{SN[chain]}", fontsize=35)
        ax.text(0.5, 0.98, "ab"[col], transform=ax.transAxes, fontsize=48, fontweight="bold", ha="center", va="top")
        ax.set_ylim(0, 0.38)
        style_axes(ax, size=35)
        ax.set_xticks(tick_carbons, [f"C{c}" for c in tick_carbons], fontsize=35, fontweight="bold")
        ax.set_xticks(carbons[carbons % 2 == 1], minor=True)
        ax.legend(fontsize=35, frameon=False)
        ax.set_xlabel("Carbon Atom", fontsize=35)
        if col == 0:
            ax.set_ylabel(r"$\mathbf{S_{CH}}$", fontsize=45)
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "reference_dopc_pure_dopc_60_op_profiles.png"), dpi=150)
    plt.show()